# Compile-time results — faithful (`new`) vs original (`old`)

Run this from `compile_time/`. It aggregates **every run** under `results/run*/` (each run folder has `new.csv` + `old.csv`), computing mean±std per prop over the runs.

Story = **bounded worst-case vs unbounded tail**: every faithful proof compiles in a few minutes; some original proofs never finish within the cap. `T.O.` = did not finish within the wall cap (censored, a lower bound), so it's plotted at the cap and hatched — not treated as a true time.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

RESULTS = Path('results')   # notebook lives in compile_time/
SCALE = 'linear'           # 'linear' (shows the gap in real time) or 'log' (shows per-prop detail)
rows = []
for world in ['new', 'old']:
    for f in sorted(RESULTS.glob(f'run*/{world}.csv')):
        d = pd.read_csv(f)
        d['prop'] = d['target'].astype(str).str.zfill(2)
        d['world'] = world
        d['run'] = f.parent.name
        rows.append(d[['prop', 'world', 'run', 'wall_s', 'status']])
assert rows, f'no run*/new.csv or run*/old.csv under {RESULTS.resolve()}'
long = pd.concat(rows, ignore_index=True)
CAP = long.loc[long.status == 'T.O.', 'wall_s'].max()
if np.isnan(CAP):
    CAP = float(long.wall_s.max())
print('runs found :', sorted(long.run.unique()))
print('worlds     :', sorted(long.world.unique()))
print('cap (s)    :', CAP)
long.head()

In [ ]:
# aggregate per (world, prop): mean/std over the runs that were 'ok'; count non-ok runs
recs = []
for (world, prop), sub in long.groupby(['world', 'prop']):
    ok = sub.loc[sub.status == 'ok', 'wall_s']
    recs.append(dict(world=world, prop=prop,
                     mean=ok.mean() if len(ok) else np.nan,
                     std=ok.std(ddof=0) if len(ok) else np.nan,
                     n_ok=len(ok),
                     n_to=int((sub.status == 'T.O.').sum()),
                     n_fail=int((sub.status == 'fail').sum()),
                     n=len(sub)))
stats = pd.DataFrame(recs)
new = stats[stats.world == 'new'].drop(columns='world').set_index('prop')
old = stats[stats.world == 'old'].drop(columns='world').set_index('prop')
W = old.join(new, lsuffix='_old', rsuffix='_new').reset_index().sort_values('prop').reset_index(drop=True)
# plotting value: mean if it ever compiled, else the cap (censored); std 0 where censored/single-run
for p in ['old', 'new']:
    W[f'{p}_plot'] = W[f'mean_{p}'].fillna(CAP)
    W[f'{p}_censored'] = W[f'n_ok_{p}'].fillna(0) == 0
    W[f'std_{p}'] = W[f'std_{p}'].fillna(0)
W

## Summary

In [ ]:
for p in ['new', 'old']:
    ok = W.loc[W[f'n_ok_{p}'] > 0, f'mean_{p}']
    ncens = int((W[f'n_ok_{p}'] == 0).sum())
    print(f'{p}: compiled {len(ok)}/{len(W)} props   max_ok={ok.max():7.1f}s   median={ok.median():6.1f}s   '
          f'never-finished(>= {CAP:.0f}s)={ncens}')

new_max = W['mean_new'].max()
print(f'\nfaithful worst-case (mean): {new_max:.1f}s  (~{new_max/60:.1f} min)')
print(f'cap: {CAP:.0f}s  ->  worst-case ratio >= {CAP/new_max:.1f}x (floor; old censored is a lower bound)')
onlynew = W[(W.n_ok_new > 0) & (W.n_ok_old == 0)]
print(f'compiles as NEW but NEVER as OLD: {len(onlynew)} props -> {list(onlynew.prop)}')

## Per-prop compile time (mean ± std; hatched = never finished within cap)

In [ ]:
h = 0.4
y = np.arange(len(W))
fig, ax = plt.subplots(figsize=(9, 13))

def draw(off, p, color, label):
    cens = W[f'{p}_censored'].values
    err = np.where(cens, 0, W[f'std_{p}'].values)
    bars = ax.barh(y + off, W[f'{p}_plot'].values, height=h, color=color, label=label,
                   xerr=err, error_kw=dict(lw=0.6), edgecolor='black', linewidth=0.3)
    for b, c in zip(bars, cens):
        if c:
            b.set_hatch('///')

draw(h/2, 'old', '#4C72B0', 'old (original)')
draw(-h/2, 'new', '#DD8452', 'new (faithful)')
ax.axvline(CAP, color='red', ls=':', lw=1, label=f'cap {CAP:.0f}s')
ax.set_yticks(y); ax.set_yticklabels(W.prop); ax.invert_yaxis()
ax.set_xscale(SCALE); ax.set_xlabel('wall seconds (mean ± std over runs)')
ax.set_ylabel('proposition')
ax.set_title('Per-prop compile time: old vs new   (hatched = never finished within cap)')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

## Sorted tail (faithful stays bounded; original blows into the cap)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for p, color, label in [('old', '#4C72B0', 'old (original)'), ('new', '#DD8452', 'new (faithful)')]:
    v = np.sort(W[f'{p}_plot'].values)
    ax.plot(np.arange(1, len(v) + 1), v, marker='o', ms=3, color=color, label=label)
ax.axhline(CAP, color='red', ls=':', lw=1, label=f'cap {CAP:.0f}s (censored above)')
ax.set_yscale(SCALE); ax.set_xlabel('props, sorted by compile time'); ax.set_ylabel('wall seconds')
ax.set_title('Sorted compile times')
ax.legend()
plt.tight_layout(); plt.show()

## old vs new scatter (below the line = faithful faster; red = old censored at cap)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
both = (W.n_ok_old > 0) & (W.n_ok_new > 0)
ax.scatter(W.mean_old[both], W.mean_new[both], c='#4C72B0', label='both compiled', zorder=3)
cens = (W.n_ok_old == 0)
if cens.any():
    ax.scatter(W.old_plot[cens], W.new_plot[cens], marker='x', c='#C44E52', label='old never finished', zorder=3)
lo = float(np.nanmin([W.mean_old.min(), W.mean_new.min()])) * 0.8
hi = CAP * 1.3
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='old = new')
ax.axvline(CAP, color='red', ls=':', lw=1)
ax.set_xscale(SCALE); ax.set_yscale(SCALE); ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel('old (original) wall s'); ax.set_ylabel('new (faithful) wall s')
ax.set_title('Points below the dashed line = faithful faster')
ax.legend()
plt.tight_layout(); plt.show()